# AI_EIARI4A_2026: Lab 0 - Baseline Transformer Internals
## 1 Million Parameter Transformer Baseline (VUT Prospectus Fine-Tuning)

**Objective:** In this lab, you will explore a small Transformer architecture and perform fine-tuning on the **VUT Prospectus 2026**. This demonstrates how a generative model can learn domain-specific information even at a small scale.

### 1. Setup and Imports

In [1]:
import tensorflow as tf
from tensorflow.keras import layers
import numpy as np
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import os

print("TensorFlow version:", tf.__version__)

### 2. Building the 1M Parameter Architecture

In [2]:
def build_mini_transformer(vocab_size, seq_len=128, embed_dim=128, num_heads=4, ff_dim=512):
    inputs = layers.Input(shape=(seq_len,))
    x = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)(inputs)
    for i in range(2):
        attention_output = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim, name=f"mha_{i}"
        )(x, x)
        x = layers.LayerNormalization(epsilon=1e-6)(x + attention_output)
        ffn_output = layers.Dense(ff_dim, activation="relu")(x)
        ffn_output = layers.Dense(embed_dim)(ffn_output)
        x = layers.LayerNormalization(epsilon=1e-6)(x + ffn_output)
    outputs = layers.Dense(vocab_size, activation="softmax")(x)
    return tf.keras.Model(inputs=inputs, outputs=outputs)

vocab_size = 5000 
model = build_mini_transformer(vocab_size=vocab_size)
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy")

### 3. The Data Pipeline: Loading the VUT Prospectus

In [3]:
path_to_file = "vut_prospectus_text.txt"
with open(path_to_file, 'r', encoding='utf-8') as f:
    text = f.read()

vectorize_layer = tf.keras.layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode='int',
    output_sequence_length=129
)

chunk_size, step = 500, 100
text_chunks = [text[i : i + chunk_size] for i in range(0, len(text) - chunk_size, step)]

def prepare_dataset(texts, batch_size=32):
    ds = tf.data.Dataset.from_tensor_slices(texts)
    vectorize_layer.adapt(ds.batch(64))
    def split_input_target(chunk): return chunk[:, :-1], chunk[:, 1:]
    return ds.batch(batch_size).map(vectorize_layer).map(split_input_target).prefetch(tf.data.AUTOTUNE)

dataset = prepare_dataset(text_chunks)

### 4. Fine-Tuning the Model
**Note**: For the chatbot to work, you must train the model. 20-30 epochs are recommended.

In [4]:
model.fit(dataset, epochs=20)

### 5. The Prospectus Chatbot Interface
Run the cell below to open the interactive chat interface. 

**Pro Tip**: This model is a **Base Model** (it predicts the next word). Instead of asking a question like *"What is engineering?"*, try starting the sentence for it: *"The Faculty of Engineering is..."*

In [5]:
def generate_response(prompt, length=30, seq_len=128):
    tokens = vectorize_layer([prompt])
    if tokens.shape[1] > seq_len: tokens = tokens[:, -seq_len:]
    elif tokens.shape[1] < seq_len:
        pad = np.zeros((1, seq_len - tokens.shape[1]), dtype=np.int32)
        tokens = tf.convert_to_tensor(np.concatenate([pad, tokens.numpy()], axis=1), dtype=tf.int32)
        
    result = prompt
    vocab = vectorize_layer.get_vocabulary()
    for _ in range(length):
        preds = model.predict(tokens, verbose=0)
        next_id = np.argmax(preds[0, -1, :])
        tokens_np = np.roll(tokens.numpy(), -1, axis=1)
        tokens_np[0, -1] = next_id
        tokens = tf.convert_to_tensor(tokens_np, dtype=tf.int32)
        word = vocab[next_id]
        if word == "": continue
        result += " " + word
    return result

# Chat UI Components
output_area = widgets.Output()
input_box = widgets.Text(placeholder='Type a prompt (e.g., Admission requirements are...)', layout=widgets.Layout(width='70%'))
send_btn = widgets.Button(description='Ask AI', button_style='primary')

def on_send(b):
    with output_area:
        user_text = input_box.value
        if not user_text: return
        input_box.value = ''
        print(f"YOU: {user_text}")
        print("AI: Thinking...")
        response = generate_response(user_text)
        clear_output(wait=True)
        display(HTML(f"""
            <div style='margin-bottom:10px; padding:10px; background:#f0f0f0; border-radius:10px'><b>You:</b> {user_text}</div>
            <div style='margin-bottom:10px; padding:10px; background:#e3f2fd; border-radius:10px; border-left:5px solid #2196f3'>
                <b>VUT-Bot:</b> {response}...
            </div>
        """))

send_btn.on_click(on_send)
display(HTML("<h3>VUT Prospectus Interactive Chat</h3>"))
display(widgets.HBox([input_box, send_btn]))
display(output_area)